# Start here — what this deployment is holding

Four cells. They say which dataset is bound, what its matrix looks like, whether
the GPU is actually usable from inside this enclave, and which `obs` columns the
ported notebooks will use.

Run them before anything else: every question that makes
`01_preprocessing.ipynb` fail halfway through is answered here in under a minute.


## 1. What is bound

A deployment of this listing reads its data from one of two places and no
others: a dataset the operator attached to the **Single-cell matrices** slot, or
a file dropped into `/workspace/data` with the file browser on the left. Nothing
in this workspace downloads a corpus from the internet.


In [ ]:
import sc_workspace as scw

print(scw.describe_binding())
for entry in scw.datasets():
    print(f"  {entry.get('dataset')}  {entry.get('interface')}  {entry.get('title') or ''}")


## 2. The matrix

`load_matrix()` resolves whichever source is present. An `.h5ad` is read as it
is; a ClickHouse dataset served through the gatekeeper is assembled from its
long-form `sc_counts` / `sc_cells` / `sc_genes` tables.


In [ ]:
adata = scw.load_matrix()
print(adata)


In [ ]:
# The columns the ported notebooks will group by. `batch_key` raises here, with
# the list of columns the matrix actually has, rather than inside a Harmony call
# forty cells into 01_preprocessing.
print("obs columns:", list(adata.obs.columns))
print("batch column:", scw.batch_key(adata))
print("annotation column:", scw.label_key(adata))


## 3. The GPU

This is the cell worth running on a new deployment. RAPIDS-singlecell computes on
the card; if the card is not reachable from inside the enclave, everything below
is a CPU-only notebook server and the listing is not doing its job.


In [ ]:
import cupy as cp
import rapids_singlecell as rsc

device = cp.cuda.Device(0)
free, total = device.mem_info
properties = cp.cuda.runtime.getDeviceProperties(0)
print(f"{properties['name'].decode()}  {total / 2**30:.1f} GiB total, {free / 2**30:.1f} GiB free")
print("rapids_singlecell", rsc.__version__)


In [ ]:
# The matrix on the card, and one real computation on it. ~seconds for a few
# hundred thousand cells; if this raises a CUDA error, nothing further will work.
import time

start = time.time()
rsc.get.anndata_to_GPU(adata)
rsc.pp.filter_genes(adata, min_cells=3)
rsc.pp.normalize_total(adata, target_sum=1e4)
rsc.pp.log1p(adata)
print(f"normalised {adata.n_obs} x {adata.n_vars} on the GPU in {time.time() - start:.1f}s")
rsc.get.anndata_to_CPU(adata)


## 4. A copy that survives a restart

`/workspace` is the volume. `/workspace/derived` is where the notebooks hand
files to one another, and it is the only place a result is safe — everything else
in this container is thrown away when the pod moves.

Nothing written here leaves the deployment. There is no egress path out of this
workspace for the data: what you export, you export yourself, by downloading it
through the browser.


In [ ]:
path = scw.derived("as_loaded.h5ad")
adata.write(path)
print("wrote", path)


## Where to go next

| Notebook | What it does |
|---|---|
| `01_preprocessing.ipynb` | the blueprint's end-to-end workflow: QC, filtering, normalisation, PCA, neighbours, UMAP/t-SNE, Louvain and Leiden, Harmony batch correction, marker genes. Writes `derived/preprocessed.h5ad`. |
| `02_regulatory_activity.ipynb` | decoupler on the GPU over what 01 wrote — transcription-factor and pathway activity. |
| `03_pearson_residuals.ipynb` | the same workflow as 01 with Pearson residuals instead of log-normalisation. |

`PROVENANCE.md` says exactly what was changed in each of them and why.
